# Day 3 — Transfer Learning: Custom Image Classifier (Milestone Project)

**Phase 2 · Deep Learning Essentials · Day 3 / 3 · 🏁 Milestone**

This is the **Phase 2 milestone project**. You will build a complete image classifier the way it is done in industry — by **fine-tuning a pre-trained ResNet** on a custom dataset.

### What you will build
A working **Cats vs Dogs** classifier that achieves **>95% accuracy in under 5 minutes of training** on a GPU — something that would take days from scratch.

### Today's flow (end-to-end)
1. **Download & prepare** a small custom dataset (Cats vs Dogs subset).
2. **Explore** the data with `ImageFolder`.
3. **Load a pre-trained ResNet18** from `torchvision`.
4. **Fine-tune** it on our dataset (replace the classification head).
5. **Evaluate** with accuracy, confusion matrix, and per-class metrics.
6. **Inspect predictions** and find what the model gets wrong.
7. **Export** the model so it can be deployed.

### Why transfer learning matters
> Models like ResNet have already learned to recognise edges, textures, shapes, and object parts on ImageNet (1.2M images, 1000 classes). Re-using these features means we need only a **tiny dataset** and **a few minutes of training** to solve a new task — instead of millions of images and days of compute.

| Approach           | Data needed | Time | Accuracy |
|--------------------|-------------|------|----------|
| Train MLP scratch  | huge        | days | low      |
| Train CNN scratch  | medium-big  | hours| medium   |
| **Transfer learn** | **small**   | **minutes** | **high** |


## 1. Setup

In [ ]:
import os, time, shutil, random, urllib.request, zipfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.optim.lr_scheduler import StepLR

import torchvision
from torchvision import datasets, transforms, models

import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42); random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch :", torch.__version__)
print("Device  :", device)

## 2. Get the dataset

We'll use a small **Cats vs Dogs** subset (a few thousand images).
The cell below downloads and unpacks it into the folder structure that PyTorch's `ImageFolder` expects:

```
data/
└── cats_and_dogs_filtered/
    ├── train/
    │   ├── cats/...
    │   └── dogs/...
    └── validation/
        ├── cats/...
        └── dogs/...
```

> If the download fails (firewall / offline), point `DATA_DIR` at any folder that follows the same `class_name/*.jpg` layout — the rest of the notebook will work unchanged.


In [ ]:
DATA_ROOT = Path("./data")
DATA_ROOT.mkdir(exist_ok=True)
ZIP_PATH  = DATA_ROOT / "cats_and_dogs_filtered.zip"
DATA_DIR  = DATA_ROOT / "cats_and_dogs_filtered"

URL = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"

if not DATA_DIR.exists():
    print("Downloading dataset...")
    urllib.request.urlretrieve(URL, ZIP_PATH)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_ROOT)
    ZIP_PATH.unlink(missing_ok=True)
    print("Done.")
else:
    print("Dataset already present.")

train_dir = DATA_DIR / "train"
val_dir   = DATA_DIR / "validation"

# Quick sanity
for split, d in (("train", train_dir), ("validation", val_dir)):
    counts = {c.name: len(list(c.glob("*"))) for c in d.iterdir() if c.is_dir()}
    print(f"{split}: {counts}")

## 3. Transforms — match the pre-trained model's expectations

ResNet18 was trained on **ImageNet** with 224×224 RGB images normalised by ImageNet's per-channel mean/std. To re-use those weights successfully, our images must be preprocessed the same way.

We also apply augmentation on the **train** set only.


In [ ]:
IMG_SIZE = 224

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## 4. ImageFolder — the easiest way to load a custom dataset

`ImageFolder` automatically:
- discovers class names from the subfolder names,
- assigns integer labels (alphabetical order),
- applies your transform pipeline.


In [ ]:
train_dataset = datasets.ImageFolder(root=str(train_dir), transform=train_tf)
val_dataset   = datasets.ImageFolder(root=str(val_dir),   transform=val_tf)

CLASSES = train_dataset.classes
NUM_CLASSES = len(CLASSES)
print("Classes :", CLASSES)
print("Train   :", len(train_dataset))
print("Val     :", len(val_dataset))

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# Visualise a batch (after un-normalising)
def denorm(img):
    img = img.clone()
    for c in range(3):
        img[c] = img[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
    return img.clamp(0, 1)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(11, 6))
for ax, img, lab in zip(axes.flat, imgs, labels):
    ax.imshow(denorm(img).permute(1, 2, 0).numpy())
    ax.set_title(CLASSES[lab.item()])
    ax.axis("off")
plt.suptitle("Augmented training samples")
plt.tight_layout()
plt.show()

## 5. Load a pre-trained ResNet18

We download ResNet18 with weights pre-trained on ImageNet.

### Two transfer-learning strategies

**A. Feature extraction**
Freeze every layer except the last classifier head. Only the new head is trained.
✅ Very fast, very little data needed
❌ Slightly lower ceiling

**B. Fine-tuning**
Replace the head AND let the rest of the network update slowly (often with a smaller learning rate for the backbone).
✅ Highest accuracy
❌ Slightly slower, more data helpful

We'll use a **hybrid** strategy that's a great default:
1. Replace the final `fc` layer with one that outputs `NUM_CLASSES` logits.
2. Train *everything* but use a **smaller learning rate** so the pre-trained features aren't destroyed.


In [ ]:
# Pre-trained backbone
weights = models.ResNet18_Weights.IMAGENET1K_V1
model = models.resnet18(weights=weights)

# Inspect the original final layer
print("Original fc:", model.fc)

# Replace the head: 512 -> NUM_CLASSES
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)
print("New fc     :", model.fc)

model = model.to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params:,}")

### (Optional) — Try the *frozen* feature-extractor variant
Uncomment the cell below to freeze every layer except the classifier head, then re-run training and compare.


In [ ]:
# for name, param in model.named_parameters():
#     if not name.startswith("fc."):
#         param.requires_grad = False
# n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Trainable params after freezing: {n_params:,}")

## 6. Loss, optimizer, scheduler

A small learning rate (`1e-4`) is the rule of thumb for fine-tuning — large updates would destroy the pre-trained features.


In [ ]:
LEARNING_RATE = 1e-4
EPOCHS = 5

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = StepLR(optimizer, step_size=3, gamma=0.5)

## 7. Reusable train / eval helpers (same pattern as Day 1 & 2)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total

In [ ]:
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc = 0.0
ckpt_path = "best_resnet18_catsdogs.pth"

start = time.time()
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    va_loss, va_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    history["train_loss"].append(tr_loss); history["train_acc"].append(tr_acc)
    history["val_loss"].append(va_loss);   history["val_acc"].append(va_acc)

    flag = ""
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save(model.state_dict(), ckpt_path)
        flag = "  <-- new best, saved"

    print(f"Epoch {epoch}/{EPOCHS} | train_loss={tr_loss:.4f} acc={tr_acc:.4f} | "
          f"val_loss={va_loss:.4f} acc={va_acc:.4f}{flag}")

print(f"\nTotal training time: {time.time() - start:.1f} s")
print(f"Best validation accuracy: {best_acc:.4f}")

## 8. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"],   label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"],   label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 9. Detailed evaluation — confusion matrix, precision, recall, F1

Accuracy alone is not enough. Real-world classifiers report:
- **Precision** — of everything I predicted as class X, how much was actually X?
- **Recall** — of everything that *was* class X, how much did I catch?
- **F1** — harmonic mean of precision and recall.


In [ ]:
# Reload best checkpoint
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        all_preds.append(model(images).argmax(1).cpu())
        all_labels.append(labels)
all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

# Confusion matrix (manual — works without sklearn)
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for t, p in zip(all_labels, all_preds):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES), CLASSES)
ax.set_yticks(range(NUM_CLASSES), CLASSES)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix — validation set")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black")
plt.colorbar(im, ax=ax, fraction=0.045)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class precision / recall / F1
print(f"{'Class':<10}{'Precision':>10}{'Recall':>10}{'F1':>10}{'Support':>10}")
for i, name in enumerate(CLASSES):
    tp = cm[i, i]
    fp = cm[:, i].sum() - tp
    fn = cm[i, :].sum() - tp
    support = cm[i].sum()
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    print(f"{name:<10}{precision:>10.3f}{recall:>10.3f}{f1:>10.3f}{support:>10d}")

acc = np.trace(cm) / cm.sum()
print(f"\nOverall accuracy: {acc:.4f}")

## 10. Where does the model get it wrong?

Looking at the actual mistakes is one of the most useful debugging steps in deep learning. Often you discover that the *labels* were noisy, or that a specific kind of image is consistently hard.


In [ ]:
# Find indexes of misclassified validation images
wrong_indices = np.where(all_preds != all_labels)[0]
print(f"Mis-classified images: {len(wrong_indices)} / {len(all_labels)}")

if len(wrong_indices) > 0:
    n_show = min(8, len(wrong_indices))
    pick = np.random.choice(wrong_indices, n_show, replace=False)

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, idx in zip(axes.flat, pick):
        img, true = val_dataset[idx]
        with torch.no_grad():
            pred = model(img.unsqueeze(0).to(device)).argmax(1).item()
        ax.imshow(denorm(img).permute(1, 2, 0).numpy())
        ax.set_title(f"P:{CLASSES[pred]} | T:{CLASSES[true]}", color="red")
        ax.axis("off")
    plt.suptitle("A sample of misclassified images")
    plt.tight_layout(); plt.show()
else:
    print("No misclassifications! 🎉")

## 11. Inference on a single image (deployment-ready)

This is exactly what you'd put behind an API endpoint or use inside a desktop app.


In [ ]:
from PIL import Image

def predict_image(model, image_path, transform, classes, device, top_k=2):
    img = Image.open(image_path).convert("RGB")
    x = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs  = torch.softmax(logits, dim=1)[0]
        topk_p, topk_i = torch.topk(probs, k=min(top_k, len(classes)))

    print(f"\nImage: {image_path}")
    for p, i in zip(topk_p.cpu().numpy(), topk_i.cpu().numpy()):
        print(f"  {classes[i]:>10s} : {p:.4f}")
    return classes[topk_i[0].item()], topk_p[0].item()

# Demo on the first validation image
demo_class_dir = next((val_dir / c) for c in CLASSES)
demo_path = next(demo_class_dir.glob("*"))
predict_image(model, str(demo_path), val_tf, CLASSES, device)

## 12. Export the model (state_dict + TorchScript)

Two common export formats:

| Format        | Use-case                                                              |
|---------------|----------------------------------------------------------------------|
| **state_dict (.pth)** | Reload in PyTorch (need the model class definition).         |
| **TorchScript (.pt)** | Standalone, runs without the original Python code (production). |

Phase 5 will cover ONNX, TensorRT and FastAPI — but TorchScript is already a great deployment artifact.


In [ ]:
# 1) Save weights only
torch.save(model.state_dict(), "resnet18_catsdogs_final.pth")

# 2) Save TorchScript (traced) — fully self-contained
model.eval()
example = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
traced = torch.jit.trace(model, example)
traced.save("resnet18_catsdogs_traced.pt")

# 3) Save the class names alongside the model
import json
with open("class_names.json", "w") as f:
    json.dump(CLASSES, f)

print("Saved:")
for fname in ["resnet18_catsdogs_final.pth",
              "resnet18_catsdogs_traced.pt",
              "class_names.json"]:
    size_mb = os.path.getsize(fname) / 1e6
    print(f"  {fname}  ({size_mb:.1f} MB)")

In [ ]:
# Demonstrate that the TorchScript model loads & runs without the SmallCNN class definition
loaded = torch.jit.load("resnet18_catsdogs_traced.pt", map_location=device)
loaded.eval()
with torch.no_grad():
    out = loaded(example)
print("TorchScript model output shape:", out.shape)

## 13. Phase 2 recap — milestone complete 🏁

Across three days you went from raw tensors to a production-ready custom image classifier.

| Day | Project                                        | Key idea                                            |
|-----|------------------------------------------------|-----------------------------------------------------|
| 1   | MLP on MNIST                                   | Tensors, autograd, training loop                    |
| 2   | CNN on CIFAR-10 with BN/Dropout/LR scheduler   | Convolutions + modern training tricks               |
| 3   | Fine-tune ResNet18 on Cats vs Dogs             | **Transfer learning** — the real-world recipe       |

### What you can now do unaided
- Load any custom image dataset organised by folder.
- Choose between *training from scratch*, *feature extraction* and *fine-tuning*.
- Apply augmentation appropriate for the train set only.
- Train, evaluate, plot curves, build a confusion matrix and per-class metrics.
- Export the model as state_dict and TorchScript.

---

### Stretch goals for tonight
1. **Swap the backbone**: replace `models.resnet18` with `models.resnet50` or `models.efficientnet_b0`. Compare accuracy and training time.
2. **Use your own dataset**: any folder of images organised as `class_name/*.jpg` will work — try a 3-class problem of your choosing.
3. **Confidence threshold**: in `predict_image`, refuse to predict when top-class probability < 0.7 — useful for real-world apps.
4. **Add more augmentation**: experiment with `RandomRotation`, `RandomAffine`, `RandAugment`.

Next up — **Phase 3: Object Detection, Segmentation, and Real-time CV.**
